# Climate Champions: Data Validation

This notebook validates eight CSV datasets exported from the Climate
Champions pre/post assessment workbook.

All eight datasets loaded successfully. The combined participant table
contains 129 track participation records, the long-format response table
contains 1,290 answer entries, and the question-reference table contains
50 entries.

Validation covered dataset sizes, missing values, duplicate keys,
consistency between track and combined files, answer formats,
answer-key formats, score arithmetic, and response-table completeness.

All structural and arithmetic checks passed. Eight responses recorded
as X and two Arduino Q5 answer-key entries recorded as ABCD require
clarification before affected scores are treated as final.

In [1]:
%pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the project from either the root or notebooks folder.
current = Path.cwd().resolve()

ROOT = next(
    (
        folder
        for folder in [current, *current.parents]
        if (folder / "data" / "processed").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Cannot find data/processed. Open your climate_champions_analysis "
        "project and place the CSV files inside data/processed."
    )

DATA = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

print("Project folder:", ROOT)
print("CSV folder:", DATA)

Project folder: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\OCTOBER_3_DATA
CSV folder: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\OCTOBER_3_DATA\data\processed


In [6]:
track_files = {
    "teachers": "teachers.csv",
    "scratch": "scratch.csv",
    "microbit_heat": "microbit_heat.csv",
    "microbit_flood": "microbit_flood.csv",
    "arduino": "arduino.csv",
}

required_files = list(track_files.values()) + [
    "participants_all.csv",
    "responses_long.csv",
    "question_reference.csv",
]

missing_files = [
    name for name in required_files
    if not (DATA / name).exists()
]

if missing_files:
    raise FileNotFoundError(f"Missing CSV files: {missing_files}")

def load_csv(filename):
    # Preserve IDs as text, including any leading zeros.
    return pd.read_csv(
        DATA / filename,
        dtype={"participant_id": "string"},
        encoding="utf-8-sig"
    )

tracks = {
    track: load_csv(filename)
    for track, filename in track_files.items()
}

participants = load_csv("participants_all.csv")
responses = load_csv("responses_long.csv")
questions = load_csv("question_reference.csv")

print("All eight datasets loaded.")

All eight datasets loaded.


## 1. Dataset Sizes and Structure

All five track datasets matched their expected record counts.

| Track | Records |
|---|---:|
| Teachers | 29 |
| Scratch | 22 |
| micro:bit Heat Alert | 25 |
| micro:bit Flood Alert | 26 |
| Arduino | 27 |
| **Total** | **129** |

Each track dataset contains 20 columns. The combined participant table
contains 129 rows and 20 columns, the response table contains 1,290 rows
and 10 columns, and the question-reference table contains 50 rows
and 7 columns.

The validation found:
- No missing values in the combined participant table.
- No missing track or participant identifiers.
- No duplicate track–participant keys.
- No blank pre/post answer cells.
- No participant IDs appearing in multiple tracks.
- An exact match between the combined participant table and the five
  track files, with data types excluded from the comparison.

The absence of repeated IDs does not independently establish that each
person received only one ID. Counts are therefore reported as track
participation records.

The combined table already contains the five track datasets. Combining
it with those files again would double-count records.

In [7]:
dataset_sizes = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1]
    }
    for name, df in {
        **tracks,
        "participants_all": participants,
        "responses_long": responses,
        "question_reference": questions,
    }.items()
])

display(dataset_sizes)

# Expected counts from the supplied workbook.
expected_counts = {
    "teachers": 29,
    "scratch": 22,
    "microbit_heat": 25,
    "microbit_flood": 26,
    "arduino": 27,
}

count_check = pd.DataFrame([
    {
        "track": track,
        "expected_records": expected,
        "loaded_records": len(tracks[track]),
        "matches_expected": len(tracks[track]) == expected
    }
    for track, expected in expected_counts.items()
])

display(count_check)

,dataset,rows,columns
0,teachers,29,20
1,scratch,22,20
2,microbit_heat,25,20
3,microbit_flood,26,20
4,arduino,27,20
5,participants_all,129,20
6,responses_long,1290,10
7,question_reference,50,7


,track,expected_records,loaded_records,matches_expected
0,teachers,29,29,True
1,scratch,22,22,True
2,microbit_heat,25,25,True
3,microbit_flood,26,26,True
4,arduino,27,27,True


In [8]:
key = ["track", "participant_id"]
answer_columns = [
    f"{phase}_q{question}"
    for phase in ["pre", "post"]
    for question in range(1, 6)
]

missing_summary = (
    participants.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(participants) * 100
).round(2)

display(
    missing_summary.loc[
        missing_summary["missing_count"] > 0
    ]
)

duplicate_records = participants.loc[
    participants.duplicated(key, keep=False)
]

print("Records with missing keys:",
      participants[key].isna().any(axis=1).sum())
print("Rows involved in duplicate keys:", len(duplicate_records))
print("Missing pre/post answers:",
      participants[answer_columns].isna().sum().sum())

# Repeated IDs across tracks need review; do not delete automatically.
id_track_counts = participants.groupby("participant_id")["track"].nunique()

print(
    "IDs appearing in multiple tracks:",
    id_track_counts.gt(1).sum()
)

,missing_count,missing_percent


Records with missing keys: 0
Rows involved in duplicate keys: 0
Missing pre/post answers: 0
IDs appearing in multiple tracks: 0


In [9]:
rebuilt_participants = pd.concat(
    tracks.values(),
    ignore_index=True
)

def sorted_records(df):
    return (
        df.sort_values(key)
        .reset_index(drop=True)
        .sort_index(axis=1)
    )

pd.testing.assert_frame_equal(
    sorted_records(rebuilt_participants),
    sorted_records(participants),
    check_dtype=False
)

print("PASS: participants_all matches the five track files.")

PASS: participants_all matches the five track files.


## 2. Answer and Answer-Key Findings

Eight of the 1,290 answer entries contain X rather than A, B, C, or D.
These represent approximately 0.62% of all answer entries.

| Track | Answer field | X entries |
|---|---|---:|
| Arduino | Pre Q4 | 1 |
| Arduino | Pre Q5 | 2 |
| micro:bit Flood Alert | Post Q1 | 1 |
| micro:bit Flood Alert | Post Q2 | 1 |
| micro:bit Flood Alert | Post Q3 | 1 |
| micro:bit Flood Alert | Post Q4 | 1 |
| micro:bit Flood Alert | Post Q5 | 1 |
| **Total** | | **8** |

These are counts of answer entries, not necessarily counts of affected
participants. X is stored as text, so it is not detected as a missing
value by the earlier missing-value check.

Two question-reference entries also require clarification:
- Arduino pre-test Q5: ABCD
- Arduino post-test Q5: ABCD

The meaning of X and the scoring rule for ABCD have not been established.
The original values are retained, and no replacement scoring rule has
been assumed.

Teachers, Scratch, and micro:bit Heat Alert had no answers outside A–D
under this check. This confirms answer format, not scoring correctness.

In [10]:
answer_audit = participants[
    key + answer_columns
].melt(
    id_vars=key,
    var_name="question_column",
    value_name="answer_raw"
)

answer_audit["answer_normalized"] = (
    answer_audit["answer_raw"]
    .astype("string")
    .str.strip()
    .str.upper()
)

valid_answers = ["A", "B", "C", "D"]

unusual_answers = answer_audit.loc[
    ~answer_audit["answer_normalized"].isin(valid_answers)
].copy()

print("Answers requiring review:", len(unusual_answers))

unusual_summary = (
    unusual_answers.groupby(
        ["track", "question_column", "answer_normalized"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)

display(unusual_summary)

Answers requiring review: 8


,track,question_column,answer_normalized,count
0,arduino,pre_q4,X,1
1,arduino,pre_q5,X,2
2,microbit_flood,post_q1,X,1
3,microbit_flood,post_q2,X,1
4,microbit_flood,post_q3,X,1
5,microbit_flood,post_q4,X,1
6,microbit_flood,post_q5,X,1


In [11]:
key_normalized = (
    questions["answer_key_raw"]
    .astype("string")
    .str.strip()
    .str.upper()
)

ambiguous_keys = questions.loc[
    ~key_normalized.isin(valid_answers),
    ["track", "phase", "question", "answer_key_raw", "skill"]
].copy()

display(ambiguous_keys)

,track,phase,question,answer_key_raw,skill
44,arduino,pre,q5,ABCD,Computational thinking: decomposition & reusab...
49,arduino,post,q5,ABCD,Computational thinking: pattern generalization...


## 3. Score Arithmetic and Response Completeness

All 129 participant records passed each of the four score checks.

| Check | Passed | Failed |
|---|---:|---:|
| Pre-test percentage matches correct count / 5 × 100 | 129 | 0 |
| Post-test percentage matches correct count / 5 × 100 | 129 | 0 |
| Reported change matches post percentage minus pre percentage | 129 | 0 |
| Pre/post correct counts are integers from 0 to 5 | 129 | 0 |

Percentage scores use a 0–100 scale. Changes are expressed in percentage
points.

The long-format response table also passed its completeness checks:
- No duplicate track–participant–phase–question keys.
- Exactly ten answer rows per participant record.
- The expected five pre-test and five post-test question slots for every
  participant record.
- Matching participant keys between the participant and response tables.

These results confirm internal arithmetic and structural consistency.
They do not independently verify the answer keys, the scoring of X,
or agreement between recorded answers and workbook correct counts.

In [12]:
score_audit = participants[
    key + [
        "pre_correct_workbook",
        "post_correct_workbook",
        "pre_score_pct_workbook",
        "post_score_pct_workbook",
        "change_pp_workbook",
    ]
].copy()

score_audit["pre_percentage_matches"] = np.isclose(
    score_audit["pre_score_pct_workbook"],
    score_audit["pre_correct_workbook"] / 5 * 100
)

score_audit["post_percentage_matches"] = np.isclose(
    score_audit["post_score_pct_workbook"],
    score_audit["post_correct_workbook"] / 5 * 100
)

score_audit["change_matches"] = np.isclose(
    score_audit["change_pp_workbook"],
    score_audit["post_score_pct_workbook"]
    - score_audit["pre_score_pct_workbook"]
)

score_audit["correct_counts_valid"] = (
    score_audit["pre_correct_workbook"].isin(range(6))
    & score_audit["post_correct_workbook"].isin(range(6))
)

check_columns = [
    "pre_percentage_matches",
    "post_percentage_matches",
    "change_matches",
    "correct_counts_valid",
]

arithmetic_summary = pd.DataFrame({
    "check": check_columns,
    "records_passed": [
        score_audit[col].sum() for col in check_columns
    ],
    "records_failed": [
        (~score_audit[col]).sum() for col in check_columns
    ],
})

display(arithmetic_summary)

,check,records_passed,records_failed
0,pre_percentage_matches,129,0
1,post_percentage_matches,129,0
2,change_matches,129,0
3,correct_counts_valid,129,0


In [13]:
response_key = [
    "track", "participant_id", "phase", "question"
]

duplicate_answer_rows = responses.duplicated(
    response_key, keep=False
).sum()

answers_per_record = (
    responses.groupby(key)
    .size()
    .rename("answer_rows")
    .reset_index()
)

expected_slots = {
    (phase, f"q{q}")
    for phase in ["pre", "post"]
    for q in range(1, 6)
}

incorrect_slot_sets = []

for record_key, group in responses.groupby(key):
    actual_slots = set(zip(group["phase"], group["question"]))

    if actual_slots != expected_slots:
        incorrect_slot_sets.append(record_key)

print("Duplicate answer rows:", duplicate_answer_rows)
print(
    "Records without exactly 10 answer rows:",
    answers_per_record["answer_rows"].ne(10).sum()
)
print("Records with incorrect question slots:", len(incorrect_slot_sets))

participant_keys = set(
    participants[key].itertuples(index=False, name=None)
)
response_keys = set(
    responses[key].itertuples(index=False, name=None)
)

print("Participant keys match:", participant_keys == response_keys)

Duplicate answer rows: 0
Records without exactly 10 answer rows: 0
Records with incorrect question slots: 0
Participant keys match: True


In [14]:
dataset_sizes.to_csv(
    TABLES / "validation_dataset_sizes.csv", index=False
)

count_check.to_csv(
    TABLES / "validation_track_counts.csv", index=False
)

unusual_summary.to_csv(
    TABLES / "validation_unusual_answers.csv", index=False
)

ambiguous_keys.to_csv(
    TABLES / "validation_ambiguous_keys.csv", index=False
)

arithmetic_summary.to_csv(
    TABLES / "validation_score_arithmetic.csv", index=False
)

print("Validation summaries saved to:", TABLES)

Validation summaries saved to: C:\Users\User\Desktop\DEVCON-DATA-ANALYTICS\OCTOBER_3_DATA\outputs\tables


## Validation Summary and Next Steps

The eight datasets loaded successfully and passed the implemented
structural, completeness, and score-arithmetic checks. All track counts
matched expectations, and the combined participant table matched the
five individual track files.

The remaining scoring questions are concentrated in Arduino and
micro:bit Flood Alert:
1. Confirm the meaning and scoring treatment of the eight X responses.
2. Confirm the intended scoring rule for Arduino Q5, whose pre/post keys
   are recorded as ABCD.
3. Confirm that example entries in the original workbook were replaced
   with actual participant records.

After clarification, recompute scores from the recorded answers and
approved answer keys, then compare them with the workbook scores.
Any corrections should be documented and applied consistently to
the track and combined datasets.

Descriptive analysis can begin for tracks without flagged answer-format
or answer-key issues, while noting that these checks do not establish
full scoring validity.

According to the source workbook, Scratch and both micro:bit tracks use
identical pre/post questions. Teachers and Arduino use different
questions, so their results should not be interpreted as matched
improvement on identical items.

Prediction analysis will use pre-test information as inputs and
post-test scores as targets. Evaluation must use participants excluded
from model training and compare model performance with simple baselines.

Five validation summary tables were exported to outputs/tables.